# PyMC Ground Truth: Joint Distributions

This notebook extends `pymc_ground_truth.ipynb` to compute Bayesian estimates for all subgroups defined in `nhanes_ground_truth.json`.

## Groups
- Overall (all data)
- Single variables (RIAGENDR, RIDAGEYR, RIDRETH1, etc.)
- Two-variable combinations (RIAGENDR x RIDAGEYR, etc.)
- Three-variable combinations (RIAGENDR x RIDAGEYR x RIDRETH1)
- Full combinations (6 variables)

In [5]:
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
from tqdm import tqdm

sys.path.append("../..")
from src.data.nhanes import load_nhanes_data

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
MIN_SAMPLES = 50

print(f"PyMC version: {pm.__version__}")

PyMC version: 5.28.0


## 1. Load and Preprocess Data

In [6]:
# Load NHANES data
df_nhanes = load_nhanes_data()
print(f"Total records: {len(df_nhanes)}")

# Keep only valid height/weight
df = df_nhanes.dropna(subset=['BMXHT', 'BMXWT']).copy()
df = df.drop_duplicates(subset=['SEQN'])
print(f"Valid subjects: {len(df)}")

Total records: 6337
Valid subjects: 6235


In [7]:
# Create categorical variables (same as nhanes_joint_distribution.ipynb)

# 1. RIDAGEYR: 5 bins
df["age_bin"] = pd.qcut(df["RIDAGEYR"], q=5, labels=False, duplicates='drop')
age_ranges = df.groupby("age_bin", observed=True)["RIDAGEYR"].agg(["min", "max"])
age_labels = {idx: f"{int(row['min'])}-{int(row['max'])}" for idx, row in age_ranges.iterrows()}
df["RIDAGEYR_cat"] = df["age_bin"].map(age_labels)

# 2. RIAGENDR
df["RIAGENDR_cat"] = df["RIAGENDR"].map({1.0: "Male", 2.0: "Female"})

# 3. RIDRETH1
race_map = {1.0: "MexicanAmerican", 2.0: "OtherHispanic", 3.0: "White", 4.0: "Black", 5.0: "Other"}
df["RIDRETH1_cat"] = df["RIDRETH1"].map(race_map)

# 4. DMDEDUC2
edu_map = {1.0: "LessThan9th", 2.0: "9thTo11th", 3.0: "HighSchool", 4.0: "SomeCollege", 5.0: "CollegeGrad"}
df["DMDEDUC2_cat"] = df["DMDEDUC2"].map(edu_map)

# 5. INDFMPIR
def map_income(x):
    if pd.isna(x): return None
    elif x < 1: return "BelowPoverty"
    elif x < 2.5: return "LowIncome"
    elif x < 4: return "MiddleIncome"
    else: return "HighIncome"
df["INDFMPIR_cat"] = df["INDFMPIR"].apply(map_income)

# 6. OCD150
activity_map = {1.0: "Sedentary", 2.0: "Light", 3.0: "Moderate", 4.0: "Heavy"}
df["OCD150_cat"] = df["OCD150"].map(activity_map)

# 7. SMQ020
df["SMQ020_cat"] = df["SMQ020"].map({1.0: "Yes", 2.0: "No"})

# Variable mapping
var_mapping = {
    "RIDAGEYR": "RIDAGEYR_cat",
    "RIAGENDR": "RIAGENDR_cat",
    "RIDRETH1": "RIDRETH1_cat",
    "DMDEDUC2": "DMDEDUC2_cat",
    "INDFMPIR": "INDFMPIR_cat",
    "OCD150": "OCD150_cat",
    "SMQ020": "SMQ020_cat"
}

print("Preprocessing complete.")
print(f"Columns: {list(var_mapping.keys())}")

Preprocessing complete.
Columns: ['RIDAGEYR', 'RIAGENDR', 'RIDRETH1', 'DMDEDUC2', 'INDFMPIR', 'OCD150', 'SMQ020']


## 2. PyMC Estimation Functions

In [8]:
def estimate_with_pymc(data, var_name="height", draws=1000, tune=500, chains=2):
    """
    Estimate normal distribution parameters using PyMC.
    
    Args:
        data: numpy array of observations
        var_name: 'height' or 'weight'
        draws, tune, chains: MCMC parameters
    
    Returns:
        dict with mu, sigma, and HDI
    """
    if len(data) < MIN_SAMPLES:
        return None
    
    # Set priors based on variable
    if var_name == "height":
        mu_prior, mu_sigma = 170, 20
        sigma_prior = 15
    else:  # weight
        mu_prior, mu_sigma = 80, 30
        sigma_prior = 30
    
    with pm.Model():
        mu = pm.Normal("mu", mu=mu_prior, sigma=mu_sigma)
        sigma = pm.HalfNormal("sigma", sigma=sigma_prior)
        obs = pm.Normal("obs", mu=mu, sigma=sigma, observed=data)
        
        trace = pm.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            random_seed=RANDOM_SEED,
            return_inferencedata=True,
            progressbar=False
        )
    
    # Extract posterior statistics
    mu_samples = trace.posterior["mu"].values.flatten()
    sigma_samples = trace.posterior["sigma"].values.flatten()
    
    return {
        "mu": round(float(np.mean(mu_samples)), 2),
        "sigma": round(float(np.mean(sigma_samples)), 2),
        "mu_hdi_94": [round(float(np.percentile(mu_samples, 3)), 2),
                      round(float(np.percentile(mu_samples, 97)), 2)],
        "sigma_hdi_94": [round(float(np.percentile(sigma_samples, 3)), 2),
                         round(float(np.percentile(sigma_samples, 97)), 2)]
    }


def estimate_group(subset_df):
    """
    Estimate height and weight distributions for a subset.
    
    Returns:
        dict with height and weight estimates, or None if insufficient data
    """
    if len(subset_df) < MIN_SAMPLES:
        return None
    
    height_data = subset_df["BMXHT"].dropna().values
    weight_data = subset_df["BMXWT"].dropna().values
    
    if len(height_data) < MIN_SAMPLES or len(weight_data) < MIN_SAMPLES:
        return None
    
    height_est = estimate_with_pymc(height_data, "height")
    weight_est = estimate_with_pymc(weight_data, "weight")
    
    if height_est is None or weight_est is None:
        return None
    
    return {
        "height": {
            "distribution_type": "normal",
            "mu": height_est["mu"],
            "sigma": height_est["sigma"],
            "unit": "cm",
            "mu_hdi_94": height_est["mu_hdi_94"],
            "sigma_hdi_94": height_est["sigma_hdi_94"]
        },
        "weight": {
            "distribution_type": "normal",
            "mu": weight_est["mu"],
            "sigma": weight_est["sigma"],
            "unit": "kg",
            "mu_hdi_94": weight_est["mu_hdi_94"],
            "sigma_hdi_94": weight_est["sigma_hdi_94"]
        },
        "n": int(len(subset_df))
    }


def make_key(var_value_pairs):
    """Create key in format: RIDAGEYR=20-39__RIAGENDR=Male__..."""
    return "__".join([f"{var}={val}" for var, val in var_value_pairs])


def parse_key(key):
    """Parse key back to variable-value pairs."""
    if key == "Overall":
        return []
    pairs = []
    for part in key.split("__"):
        var, val = part.split("=")
        pairs.append((var, val))
    return pairs


def filter_df(df, var_value_pairs):
    """Filter dataframe by variable-value pairs."""
    if not var_value_pairs:
        return df
    
    mask = pd.Series([True] * len(df), index=df.index)
    for var, val in var_value_pairs:
        cat_col = var_mapping[var]
        mask &= (df[cat_col] == val)
    return df[mask]


print("Functions defined.")

Functions defined.


## 3. Load Existing Keys from nhanes_ground_truth.json

In [9]:
# Load existing ground truth to get all keys
with open("../../data/processed/nhanes_ground_truth.json", "r") as f:
    existing_gt = json.load(f)

all_keys = list(existing_gt["distributions"].keys())
print(f"Total keys to process: {len(all_keys)}")
print(f"\nSample keys:")
for k in all_keys[:5]:
    print(f"  {k}")
print("  ...")

Total keys to process: 157

Sample keys:
  Overall
  RIDAGEYR=34-48
  RIDAGEYR=62-70
  RIDAGEYR=18-33
  RIDAGEYR=49-61
  ...


## 4. Run PyMC Estimation for All Groups

This will take several minutes as we run MCMC for each group.

In [10]:
# Run estimation for all groups
distributions = {}
failed_keys = []

print(f"Processing {len(all_keys)} groups...")
print("This may take 10-20 minutes.\n")

for key in tqdm(all_keys):
    try:
        # Parse key and filter data
        var_value_pairs = parse_key(key)
        subset = filter_df(df, var_value_pairs)
        
        # Estimate
        result = estimate_group(subset)
        
        if result is not None:
            distributions[key] = result
        else:
            failed_keys.append((key, "insufficient data"))
            
    except Exception as e:
        failed_keys.append((key, str(e)))

print(f"\nCompleted: {len(distributions)} / {len(all_keys)} groups")
if failed_keys:
    print(f"Failed: {len(failed_keys)} groups")

Processing 157 groups...
This may take 10-20 minutes.



  0%|          | 0/157 [00:00<?, ?it/s]Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [mu, sigma]
Sampling 2 chains for 500 tune and 1_000 draw iterations (1_000 + 2_000 draws total) took 0 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [mu, sigma]
Sampling 2 chains for 500 tune and 1_000 draw iterations (1_000 + 2_000 draws total) took 0 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
  1%|          | 1/157 [00:01<03:37,  1.40s/it]Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [mu, sigma]
Sampling 2 chains for 500 tune and 1_000 draw iterations (1_000 + 2_000 draws total) took 0 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Initializing NUTS using ji


Completed: 157 / 157 groups


In [11]:
# Show failed keys if any
if failed_keys:
    print("Failed keys:")
    for key, reason in failed_keys[:10]:
        print(f"  {key}: {reason}")
    if len(failed_keys) > 10:
        print(f"  ... and {len(failed_keys) - 10} more")

## 5. Preview Results

In [12]:
# Preview some results
print("Sample Results:")
print("=" * 70)

sample_keys = ["Overall", "RIAGENDR=Male", "RIAGENDR=Female", 
               "RIAGENDR=Male__RIDAGEYR=18-33", "RIAGENDR=Female__RIDAGEYR=71-80"]

for key in sample_keys:
    if key in distributions:
        d = distributions[key]
        print(f"\n{key} (N={d['n']})")
        print(f"  Height: {d['height']['mu']} +/- {d['height']['sigma']} cm")
        print(f"          94% HDI: {d['height']['mu_hdi_94']}")
        print(f"  Weight: {d['weight']['mu']} +/- {d['weight']['sigma']} kg")
        print(f"          94% HDI: {d['weight']['mu_hdi_94']}")

Sample Results:

Overall (N=6235)
  Height: 166.97 +/- 10.03 cm
          94% HDI: [166.73, 167.21]
  Weight: 82.87 +/- 22.45 kg
          94% HDI: [82.33, 83.41]

RIAGENDR=Male (N=2817)
  Height: 174.47 +/- 7.73 cm
          94% HDI: [174.2, 174.74]
  Weight: 88.79 +/- 21.85 kg
          94% HDI: [88.04, 89.58]

RIAGENDR=Female (N=3418)
  Height: 160.79 +/- 7.06 cm
          94% HDI: [160.57, 161.01]
  Weight: 78.0 +/- 21.77 kg
          94% HDI: [77.31, 78.69]

RIAGENDR=Male__RIDAGEYR=18-33 (N=600)
  Height: 176.07 +/- 7.68 cm
          94% HDI: [175.48, 176.64]
  Weight: 85.57 +/- 23.33 kg
          94% HDI: [83.85, 87.34]

RIAGENDR=Female__RIDAGEYR=71-80 (N=582)
  Height: 157.57 +/- 6.67 cm
          94% HDI: [157.06, 158.1]
  Weight: 72.79 +/- 17.29 kg
          94% HDI: [71.42, 74.21]


## 6. Compare with Frequentist Estimates

In [13]:
# Compare PyMC vs simple statistics for a few groups
print("Comparison: PyMC Bayesian vs Frequentist (Sample Statistics)")
print("=" * 80)

comparison_keys = ["Overall", "RIAGENDR=Male", "RIAGENDR=Female"]

for key in comparison_keys:
    if key not in distributions:
        continue
        
    # Get frequentist stats from existing file
    freq = existing_gt["distributions"][key]
    bayes = distributions[key]
    
    print(f"\n{key}:")
    print(f"  Height mu:  Freq={freq['height_mean']:.2f}, Bayes={bayes['height']['mu']:.2f}")
    print(f"  Height std: Freq={freq['height_std']:.2f}, Bayes={bayes['height']['sigma']:.2f}")
    print(f"  Weight mu:  Freq={freq['weight_mean']:.2f}, Bayes={bayes['weight']['mu']:.2f}")
    print(f"  Weight std: Freq={freq['weight_std']:.2f}, Bayes={bayes['weight']['sigma']:.2f}")

Comparison: PyMC Bayesian vs Frequentist (Sample Statistics)

Overall:
  Height mu:  Freq=166.97, Bayes=166.97
  Height std: Freq=10.03, Bayes=10.03
  Weight mu:  Freq=82.87, Bayes=82.87
  Weight std: Freq=22.45, Bayes=22.45

RIAGENDR=Male:
  Height mu:  Freq=174.47, Bayes=174.47
  Height std: Freq=7.73, Bayes=7.73
  Weight mu:  Freq=88.78, Bayes=88.79
  Weight std: Freq=21.84, Bayes=21.85

RIAGENDR=Female:
  Height mu:  Freq=160.79, Bayes=160.79
  Height std: Freq=7.05, Bayes=7.06
  Weight mu:  Freq=77.99, Bayes=78.00
  Weight std: Freq=21.76, Bayes=21.77


## 7. Save Results

In [14]:
# Prepare output
output = {
    "metadata": {
        "source": "NHANES August2021-August2023",
        "method": "PyMC Bayesian Estimation",
        "description": "Ground truth distributions for height (cm) and weight (kg)",
        "pymc_version": pm.__version__,
        "min_samples_per_entry": MIN_SAMPLES,
        "sampling": {
            "draws": 1000,
            "tune": 500,
            "chains": 2,
            "random_seed": RANDOM_SEED
        },
        "priors": {
            "height": {"mu": "Normal(170, 20)", "sigma": "HalfNormal(15)"},
            "weight": {"mu": "Normal(80, 30)", "sigma": "HalfNormal(30)"}
        },
        "variables": existing_gt["metadata"]["variables"],
        "key_format": "COLNAME=value__COLNAME=value"
    },
    "distributions": distributions
}

# Save
output_path = "../../data/processed/pymc_ground_truth_joint.json"
with open(output_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")
print(f"Total distributions: {len(distributions)}")

Saved to: ../../data/processed/pymc_ground_truth_joint.json
File size: 95.2 KB
Total distributions: 157
